# Start: std::variant w C++17

Tutaj przygotujesz środowisko, sprawdzisz kompilator i wybierzesz rozdział do nauki.


## Spis treści

- [Przygotowanie](#sekcja-01)
- [Jak działają komórki](#sekcja-02)
- [Sprawdzenie środowiska](#sekcja-03)
- [Kolejność czytania](#sekcja-04)
- [Artykuły i dokumentacja](#sekcja-05)
    - [Artykuł na dobry początek](#sekcja-06)
    - [Dokumentacja środowiska](#sekcja-07)


<a id="sekcja-01"></a>

## Przygotowanie

Potrzebujesz Pythona 3.11-3.14, uv oraz kompilatora i biblioteki standardowej C++17. W katalogu repozytorium wykonaj `uv sync`, a następnie `uv run jupyter lab notebooks/00_start.ipynb`. Szczegóły są w [README](../README.md). Wybierz kernel **Python 3 (ipykernel)**. Zapisane wyniki można czytać bez wykonywania komórek.

Kompilatorem może być `c++`, `clang++` albo `g++`, także w Windows. MSVC `cl` używa innych flag i nie jest obsługiwany przez pomocnik. Na macOS potrzebne są narzędzia deweloperskie, na Linux kompilator wraz z nagłówkami biblioteki standardowej, a w Windows kompilator dostępny w PATH kernela, np. z MSYS2 lub środowiska WSL.

Domyślnie wybierany jest `c++`. Aby wskazać inny kompilator, ustaw `CXX` przed uruchomieniem Jupytera albo zmień `COMPILER_NAME` w komórce przygotowującej danego notebooka. Podaj nazwę lub ścieżkę samego programu, bez flag. Homebrew GCC na macOS może wymagać `SDKROOT`: uruchom Jupytera z terminala przez `SDKROOT="$(xcrun --show-sdk-path)" CXX=g++-15 uv run jupyter lab notebooks/00_start.ipynb`.

<a id="sekcja-02"></a>

## Jak działają komórki

Uruchamiaj komórki skrótem `Shift+Enter` lub przyciskiem wykonania w JupyterLab. Pierwsza komórka Python każdego rozdziału rejestruje `%%cpp`. Każda komórka `%%cpp` jest osobnym programem C++17 i zawiera potrzebne nagłówki, definicje oraz `main()`. Po edycji uruchom ponownie tę samą komórkę. Zmienne nie przechodzą między programami; powtórzone deklaracje pozwalają wykonywać przykłady niezależnie.

Pomocnik kompiluje z `-Wall -Wextra -Wpedantic -Werror`, pokazuje wynik i usuwa pliki tymczasowe. Limit kompilacji wynosi 60 sekund, a wykonania 10 sekund. Dwie celowo błędne demonstracje w ostatnim rozdziale używają `%%cpp --expect-error`: pokazują początek komunikatu kompilatora bez przerywania całego notebooka. Jeśli taki kod skompiluje się poprawnie, pomocnik zgłosi błąd. Po poprawieniu kodu usuń `--expect-error`.

<a id="sekcja-03"></a>

## Sprawdzenie środowiska

Pierwsza komórka przygotowuje polecenie `%%cpp`. Druga wypisuje wersje Pythona i kompilatora. Trzecia kompiluje i wykonuje mały program z `<variant>`; jej wynik powinien brzmieć `C++17 i std::variant: OK`.


In [4]:
import os
import shutil
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.core.magic import register_cell_magic

COMPILER_NAME = os.environ.get("CXX", "c++")

@register_cell_magic
def cpp(line: str, cell: str) -> None:
    """Kompiluje i uruchamia jedną niezależną demonstrację C++17."""
    option = line.strip()
    if option not in {"", "--expect-error"}:
        raise ValueError("Użyj %%cpp albo %%cpp --expect-error.")
    expect_error = option == "--expect-error"
    compiler = shutil.which(COMPILER_NAME)
    if compiler is None:
        raise RuntimeError(
            f"Nie znaleziono kompilatora {COMPILER_NAME!r}. "
            "Zainstaluj kompilator C++17 lub ustaw COMPILER_NAME."
        )
    with TemporaryDirectory(prefix="variant-notebook-") as directory:
        source = Path(directory) / "example.cpp"
        executable = Path(directory) / ("example.exe" if os.name == "nt" else "example")
        source.write_text(cell, encoding="utf-8")
        compiled = subprocess.run(
            [compiler, "-std=c++17", "-Wall", "-Wextra", "-Wpedantic", "-Werror",
             str(source), "-o", str(executable)],
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=60,
        )
        if expect_error:
            if compiled.returncode == 0:
                raise RuntimeError("Oczekiwano błędu kompilacji, ale program skompilował się poprawnie.")
            print("Oczekiwany błąd kompilacji. Początek komunikatu kompilatora:")
            diagnostics = (compiled.stdout + compiled.stderr).replace(str(source), "example.cpp").splitlines()
            print("\n".join(diagnostics[:16]))
            if len(diagnostics) > 16:
                print("[Dalsza część komunikatu została pominięta.]")
            return
        if compiled.returncode != 0:
            raise RuntimeError("Błąd kompilacji:\n" + compiled.stdout + compiled.stderr)
        executed = subprocess.run(
            [str(executable)], capture_output=True, text=True,
            encoding="utf-8", errors="replace", timeout=10,
        )
        print(executed.stdout, end="")
        if executed.stderr:
            print(executed.stderr, end="")
        if executed.returncode != 0:
            raise RuntimeError(f"Program zakończył się kodem {executed.returncode}.")

print(f"Gotowe: %%cpp używa {COMPILER_NAME}, standard C++17.")

Gotowe: %%cpp używa c++, standard C++17.


In [5]:
import sys

compiler_path = shutil.which(COMPILER_NAME)
if compiler_path is None:
    raise RuntimeError(
        f"Nie znaleziono {COMPILER_NAME!r}. Sprawdź instalację kompilatora, "
        "PATH kernela lub ustaw COMPILER_NAME."
    )
compiler_info = subprocess.run(
    [compiler_path, "--version"], capture_output=True, text=True, check=True
)
print(f"Python: {sys.version.split()[0]}")
print(f"Kompilator: {compiler_path}")
print(compiler_info.stdout.splitlines()[0])


Python: 3.14.3
Kompilator: /usr/bin/c++
Apple clang version 21.0.0 (clang-2100.3.34.2)


In [6]:
%%cpp
#include <iostream>
#include <string>
#include <variant>

static_assert(__cplusplus >= 201703L, "Wymagany jest C++17 lub nowszy standard");
int main() {
    std::variant<int, std::string> value{17};
    if (std::get<int>(value) != 17) {
        return 1;
    }
    std::cout << "C++17 i std::variant: OK\n";
}


C++17 i std::variant: OK


<a id="sekcja-04"></a>

## Kolejność czytania

1. **Podstawy:** [Podstawy i odczyt](01_podstawy.ipynb) - ustawienie jako liczba albo tekst, bezpieczny dostęp i porównanie z ręczną flagą.
2. **Praktyczne użycia:** [Visit i overloaded](02_visit.ipynb) - wspólna operacja dla różnych typów oraz oddzielne lambdy.
3. **Praktyczne użycia:** [Praktyczne zastosowania](03_zastosowania.ipynb) - konfiguracja klienta, wynik parsowania i stany połączenia.
4. **Rozszerzenie dla dociekliwych:** [Pułapki i ograniczenia](04_pulapki.ipynb) - materiał rozszerzający: pusty stan, wyjątki, konwersje, pamięć i błędy kompilacji.

Każdy rozdział ma własne przygotowanie `%%cpp` i można wykonać go niezależnie. Podstawową ścieżkę tworzą rozdziały 01-03. Rozdział 04 służy do pogłębienia tematu. [Pełny przewodnik](../PRZEWODNIK.md) zawiera objaśnienia w jednym dokumencie, a [ściąga](../SCIAGA.md) zbiera najważniejsze operacje.


<a id="sekcja-05"></a>

## Artykuły i dokumentacja

<a id="sekcja-06"></a>

### Artykuł na dobry początek

- [C++ Stories: Everything You Need to Know About std::variant](https://www.cppstories.com/2018/06/variant/) - przegląd tematu i przykłady zastosowań.

<a id="sekcja-07"></a>

### Dokumentacja środowiska

- [uv: Working on projects](https://docs.astral.sh/uv/guides/projects/) - środowisko projektu i plik blokady zależności.
- [IPython: Defining custom magics](https://ipython.readthedocs.io/en/stable/config/custommagics.html) - mechanizm użyty do zdefiniowania polecenia `%%cpp`.
- [cppreference: std::variant](https://en.cppreference.com/cpp/utility/variant) - dokumentacja biblioteki, z której korzysta program kontrolny.

[Przejdź do podstaw i odczytu](01_podstawy.ipynb) | [README](../README.md)
